In [ ]:
import torch
import math

# Use float64 to match MATLAB's default double precision
dtype = torch.float64 

# Constants (assuming values from MATLAB comments)
dim = 6
dt = 0.01
total_iter = 1000
N = 100  # Number of particles
tol = 1e-5
numsteps_lambda = 50 #500
delta_lambda = 1.0/numsteps_lambda
# Sensor locations (not used in the generation loop, but translated for completeness)
s1 = torch.tensor([[10.0], [0.0]], dtype=dtype)
s2 = torch.tensor([[30.0], [0.0]], dtype=dtype)

# Target dynamics: x(n+1) = Ax(n) + Bw(n)
A = torch.zeros((dim, dim), dtype=dtype)
A[0, 2] = 1.0; A[1, 3] = 1.0
A[2, 4] = 1.0; A[3, 5] = 1.0

B = torch.zeros((dim, 2), dtype=dtype)
B[4, 0] = 1.0; B[5, 1] = 1.0

# Noise matrices
R = torch.diag(torch.tensor([0.001**2, 0.001**2], dtype=dtype))
W = torch.diag(torch.tensor([0.001**2, 0.001**2], dtype=dtype))
W_est = torch.diag(torch.tensor([0.002**2, 0.002**2], dtype=dtype))
Q = 1.0e-06*torch.eye(dim)

# Initial true target condition
x0 = torch.tensor([[5.0], [20.0], [0.0], [0.0], [2.0], [0.0]], dtype=dtype)

# Initial uncertainty
phat_0 = 1.0e-01 * torch.eye(dim, dtype=dtype)

# Define storage tensors
time = torch.zeros((1, total_iter), dtype=dtype)
xtrue = torch.zeros((dim, total_iter), dtype=dtype)
xhat = torch.zeros((dim, total_iter), dtype=dtype)
phat = torch.zeros((dim**2, total_iter), dtype=dtype)
x_error = torch.zeros((dim, total_iter), dtype=dtype)
err_norm = torch.zeros((2, total_iter), dtype=dtype)
err_3sigma = torch.zeros((2, total_iter), dtype=dtype)
Rstiff = torch.zeros((1, total_iter), dtype=dtype)

# Set initial truth state (xtrue(:,1) = x0)
xtrue[:, 0:1] = x0

# Pre-calculate discrete transition matrix A1 (Moved outside loop for efficiency)
# A1 = eye(dim) + dt*A + 0.5*A*A*dt*dt
A1 = torch.eye(dim, dtype=dtype) + (dt * A) + (0.5 * dt**2 * (A @ A))

# Pre-calculate noise scaling matrix
# sqrt(dt) * B * sqrtm(W)
# Note: Because W is a diagonal matrix, torch.sqrt acts exactly like MATLAB's sqrtm here
noise_scale = math.sqrt(dt) * B @ torch.sqrt(W)

# for ii=2:total_iter in MATLAB becomes 1 to total_iter-1 in Python (0-based indexing)
for ii in range(1, total_iter):
    time[:, ii] = time[:, ii-1] + dt
    
    # Generate random noise: randn(2,1)
    noise = torch.randn((2, 1), dtype=dtype)
    
    # State transition: xtrue(:,ii) = A1*xtrue(:,ii-1) + noise_term
    xtrue[:, ii:ii+1] = (A1 @ xtrue[:, ii-1:ii]) + (noise_scale @ noise)

In [ ]:
import torch
import math

dtype = torch.float64

def homotopy_flow(xp, xprior, P, meas, R, H, lam, s1, s2, Q0):
    dim, N = xp.shape
    tol = 1.0e-5
    
    # Calculate exact nonlinear measurements for all particles
    x1 = xp[0, :] - s1[0, 0]; y1 = xp[1, :] - s1[1, 0]
    x2 = xp[0, :] - s2[0, 0]; y2 = xp[1, :] - s2[1, 0]
    
    # hx shape is (2, N)
    hx = torch.stack([torch.atan2(y1, x1), torch.atan2(y2, x2)], dim=0) 
    
    del_z = meas - hx # Broadcasting (2, 1) to (2, N)
    del_x = xp - xprior # Broadcasting (dim, 1) to (dim, N)

    inv_R = torch.linalg.inv(R)
    inv_P = torch.linalg.inv(P)
    HtRH = H.T @ inv_R @ H
    
    dlogh_dx = H.T @ inv_R @ del_z  
    d2logh_dx2 = -HtRH
    
    dlogp_dx = -inv_P @ del_x + lam * dlogh_dx
    d2logp_dx2 = -inv_P - lam * HtRH
    
    inv_RH = torch.linalg.inv(R + lam * H @ P @ H.T)
    d2logp_dx2_inv = -(P - lam * P @ H.T @ inv_RH @ H @ P)
    
    Ad = -0.5 * d2logp_dx2_inv @ d2logh_dx2
    M = -d2logp_dx2
    
    K = 0.5 * d2logp_dx2 @ Q0 @ d2logp_dx2 + 0.5 * d2logh_dx2

    slope = d2logp_dx2_inv @ (-dlogh_dx + K @ d2logp_dx2_inv @ dlogp_dx) 
    Qh = Q0

    # Stiffness ratio
    Acoef = d2logp_dx2_inv @ (HtRH + K)
    E = torch.linalg.eigvals(Acoef)
    r = torch.max(torch.abs(E)) / torch.min(torch.abs(E))
    
    return slope, Qh, r.item()

def RicattiSolver(E, C, d2logg_dx2, d2logh_dx2, delta_lambda):
    lambda_no = int(round(1.0 / delta_lambda))
    n = E.shape[0]
    X = torch.zeros((n, n, lambda_no + 1), dtype=dtype)
    
    X[:, :, lambda_no] = E
    for iii in range(1, lambda_no + 1):
        index = lambda_no - iii + 1
        lam = index * delta_lambda
        
        d2logp_dx2 = d2logg_dx2 + lam * d2logh_dx2 
        D = -d2logp_dx2
        Ad1 = -0.5 * torch.linalg.inv(d2logp_dx2) @ d2logh_dx2
        B = 0.5 * d2logp_dx2
        X1 = X[:, :, index]
        
        df = X1 @ B @ torch.linalg.inv(D) @ B.T @ X1 - C - X1 @ Ad1.T - Ad1 @ X1
        Q = X1 - delta_lambda * df 
        Q = 0.5 * (Q + Q.T) # enforce symmetry
        
        # [V,D] = eig(Q); Q0 = V*max(D,0)*V';
        L, V = torch.linalg.eigh(Q) 
        Q0 = V @ torch.diag(torch.clamp(L, min=0.0)) @ V.T
        
        X[:, :, index - 1] = Q0
        
    return 0.5 * X

In [ ]:
# Assuming c_index is defined, defaulting to 1 based on MATLAB flow
c_index = 1 

# Re-initialize to perfectly match the block provided
xhat0 = torch.tensor([[25.0], [10.0], [0.0], [0.0], [0.0], [0.0]], dtype=dtype)
xhat[:, 0:1] = xhat0
phat[:, 0:1] = phat_0.view(dim**2, 1)

# Generate particles
particles = xhat[:, 0:1].repeat(1, N) + torch.sqrt(phat_0) @ torch.randn((dim, N), dtype=dtype)

x_error[:, 0:1] = xhat[:, 0:1] - xtrue[:, 0:1]
err_norm[0, 0] = torch.norm(x_error[0:3, 0])
err_norm[1, 0] = torch.norm(x_error[3:6, 0])
err_3sigma[0, 0] = 3.0 * math.sqrt(torch.trace(phat_0[0:3, 0:3]))
err_3sigma[1, 0] = 3.0 * math.sqrt(torch.trace(phat_0[3:6, 3:6]))

# simulation
for iter in range(total_iter):
    
    # generate measurement
    ps1 = xtrue[0:2, iter:iter+1] - s1
    ps2 = xtrue[0:2, iter:iter+1] - s2
    
    meas_clean = torch.tensor([[torch.atan2(ps1[1, 0], ps1[0, 0])],
                               [torch.atan2(ps2[1, 0], ps2[0, 0])]], dtype=dtype)
    meas = meas_clean + torch.sqrt(R) @ torch.randn((2, 1), dtype=dtype)
    
    xprior = xhat[:, iter:iter+1]
    Pprior = phat[:, iter:iter+1].view(dim, dim)
    
    ######################################
    H = torch.zeros((2, dim), dtype=dtype)
    
    x1 = xprior[0, 0] - s1[0, 0]; y1 = xprior[1, 0] - s1[1, 0]
    r1_sq = x1**2 + y1**2
    H[0, 0] = -y1 / (r1_sq + tol)
    H[0, 1] = x1 / (r1_sq + tol)
    
    x2 = xprior[0, 0] - s2[0, 0]; y2 = xprior[1, 0] - s2[1, 0]
    r2_sq = x2**2 + y2**2
    H[1, 0] = -y2 / (r2_sq + tol)
    H[1, 1] = x2 / (r2_sq + tol)
    
    ######################################
    inv_R = torch.linalg.inv(R)
    inv_P = torch.linalg.inv(Pprior)
    HtRH = H.T @ inv_R @ H
    d2logh_dx2 = -HtRH
    d2logg_dx2 = -inv_P
    
    ######################################
    # solve Recatti equation
    if c_index > 1:
        if c_index == 2:
            C = 1.0e-2 * torch.eye(dim, dtype=dtype)
            E_mat = 0.5 * 1.0e-2 * torch.eye(dim, dtype=dtype)
        elif c_index == 3:
            C = 1.0e-3 * torch.eye(dim, dtype=dtype)
            E_mat = 1.0e-3 * torch.eye(dim, dtype=dtype)
        elif c_index == 4:
            C = 5.0 * 1.0e-3 * torch.eye(dim, dtype=dtype)
            E_mat = 0.5 * 1.0e-3 * torch.eye(dim, dtype=dtype)
            
        Q = RicattiSolver(E_mat, C, d2logg_dx2, d2logh_dx2, delta_lambda)
    
    ######################################
    # particles updating
    r_vals = torch.zeros(numsteps_lambda, dtype=dtype)
    
    for i_homotopy in range(1, numsteps_lambda + 1):
        
        lam = i_homotopy * delta_lambda
        
        if c_index == 1:
            d2logp_dx2 = d2logg_dx2 + lam * d2logh_dx2
            inv_d2logp_dx2 = torch.linalg.inv(d2logp_dx2)
            Q0 = -inv_d2logp_dx2 @ d2logh_dx2 @ inv_d2logp_dx2
        else:
            Q0 = Q[:, :, i_homotopy - 1] # map 1-based logic to 0-based array index
            
        slope, Qh, r1 = homotopy_flow(particles, xprior, Pprior, meas, R, H, lam, s1, s2, Q0)
        
        # Calculate real(sqrtm(Qh+tol*eye)) safely via eigendecomposition
        Qh_safe = Qh + tol * torch.eye(dim, dtype=dtype)
        L, V = torch.linalg.eigh(Qh_safe)
        sqrtm_Qh = V @ torch.diag(torch.sqrt(torch.clamp(L, min=0.0))) @ V.T
        
        particles = particles + delta_lambda * slope + math.sqrt(delta_lambda) * sqrtm_Qh @ torch.randn((dim, N), dtype=dtype)
        
        r_vals[i_homotopy - 1] = r1
        
    # state estimate and covariance
    x_est = torch.mean(particles, dim=1, keepdim=True)
    P_est = torch.cov(particles)
    
    # prediction according to state dynamics
    A1 = torch.eye(dim, dtype=dtype) + dt * A + 0.5 * (A @ A) * dt**2
    x_pred = A1 @ x_est
    P_pred = A1 @ P_est @ A1.T + dt * B @ W_est @ B.T
    particles = A1 @ particles
    
    # put them in right places
    xhat[:, iter:iter+1] = x_est
    phat[:, iter:iter+1] = P_est.view(dim**2, 1)
    
    if iter < total_iter - 1:
        xhat[:, iter+1:iter+2] = x_pred
        phat[:, iter+1:iter+2] = P_pred.view(dim**2, 1)
        
    x_error[:, iter:iter+1] = xhat[:, iter:iter+1] - xtrue[:, iter:iter+1]
    err_norm[0, iter] = torch.norm(x_error[0:3, iter])
    err_norm[1, iter] = torch.norm(x_error[3:6, iter])
    err_3sigma[0, iter] = 3.0 * math.sqrt(torch.trace(P_est[0:3, 0:3]))
    err_3sigma[1, iter] = 3.0 * math.sqrt(torch.trace(P_est[3:6, 3:6]))
    
    Rstiff[0, iter] = torch.max(r_vals)
    
test_stop = 1

In [ ]:
torch.abs(torch.mean(xhat - xtrue))

In [ ]:
import matplotlib.pyplot as plt

# Convert PyTorch tensors to NumPy arrays for easy plotting
time_np = time.numpy().flatten()
xtrue_np = xtrue.numpy()
xhat_np = xhat.numpy()
s1_np = s1.numpy().flatten()
s2_np = s2.numpy().flatten()
err_norm_np = err_norm.numpy()
err_3sigma_np = err_3sigma.numpy()

# Set up the figure with two subplots
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ==========================================
# Plot 1: 2D Spatial Trajectory (X vs Y)
# ==========================================
ax1 = axes[0]
ax1.plot(xtrue_np[0, :], xtrue_np[1, :], label='True Trajectory', color='blue', linewidth=2)
ax1.plot(xhat_np[0, :], xhat_np[1, :], label='Inferred Trajectory', color='red', linestyle='--', linewidth=2)

# Mark the sensor locations
ax1.scatter([s1_np[0], s2_np[0]], [s1_np[1], s2_np[1]], 
            marker='^', color='green', s=150, label='Sensors', zorder=5)

# Mark the starting points
ax1.scatter(xtrue_np[0, 0], xtrue_np[1, 0], color='blue', marker='o', s=80, label='True Start', zorder=5)
ax1.scatter(xhat_np[0, 0], xhat_np[1, 0], color='red', marker='x', s=80, label='Estimate Start', zorder=5)

ax1.set_title('Target Tracking: True vs Inferred Spatial Trajectory')
ax1.set_xlabel('X Position')
ax1.set_ylabel('Y Position')
ax1.legend()
ax1.grid(True)
ax1.axis('equal') # Keeps the aspect ratio 1:1 for spatial accuracy

# ==========================================
# Plot 2: Position Error & 3-Sigma Bounds
# ==========================================
# This plots the norm of the position error (X, Y, Z) against the filter's estimated 3-sigma bounds
ax2 = axes[1]
ax2.plot(time_np, err_norm_np[0, :], label='Position Error Norm', color='black', linewidth=1.5)
ax2.plot(time_np, err_3sigma_np[0, :], label='+3 Sigma Bound', color='red', linestyle='--', linewidth=1.5)
# Lower bound is implicitly 0 for error norms, but tracking literature often plots it symmetrically
ax2.plot(time_np, -err_3sigma_np[0, :], label='-3 Sigma Bound', color='red', linestyle='--', linewidth=1.5)

ax2.set_title('Filter Consistency: Position Error vs 3-Sigma Bounds')
ax2.set_xlabel('Time (s)')
ax2.set_ylabel('Position Error Norm')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
import torch.nn as nn
import math
import torch
import pytorch_lightning as pl


# 1. Observation Encoder
class ObservationEncoder(nn.Module):
    """Processes the sliding window of sensor measurements."""
    def __init__(self, obs_dim=2, latent_dim=32):
        super().__init__()
        self.history_gru = nn.GRU(input_size=obs_dim, hidden_size=latent_dim, batch_first=True)
        self.norm = nn.LayerNorm(latent_dim)
        
    def forward(self, obs_seq):
        # Process the sequence and return the final hidden state
        _, gru_hidden = self.history_gru(obs_seq)
        return self.norm(gru_hidden.squeeze(0))

# 2. Motion Encoder
class MotionEncoder(nn.Module):
    """Processes the previously predicted state (the physics prior)."""
    def __init__(self, state_dim=6, latent_dim=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, latent_dim),
            nn.LayerNorm(latent_dim),
            nn.SiLU(),
            nn.Linear(latent_dim, latent_dim)
        )
        
    def forward(self, x_prev):
        return self.net(x_prev)

# 3. The Continuous ODE Function
class ODEFunc(nn.Module):
    """Calculates the continuous derivative for the RK4 solver."""
    def __init__(self, state_dim=6, context_dim=32, hidden_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim + context_dim + 1, hidden_dim), 
            nn.LayerNorm(hidden_dim), 
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim), 
            nn.SiLU(),
            nn.Linear(hidden_dim, state_dim) 
        )
    
    def forward(self, tau, x, context):
        tau_vec = torch.full((x.shape[0], 1), tau, device=x.device)
        h = torch.cat([x, context, tau_vec], dim=-1)
        return self.net(h).tanh()

# 4. The Lightning Module (Putting it together)
class ModularNeuralODEFilter(pl.LightningModule):
    def __init__(self, state_dim=6, obs_dim=2, context_dim=32, hidden_dim=128, lr=1e-3, noise_std=0.05):
        super().__init__()
        self.save_hyperparameters()
        self.lr = lr
        self.noise_std = noise_std # For training stability
        
        # Instantiate the modules
        self.obs_encoder = ObservationEncoder(obs_dim, context_dim)
        self.motion_encoder = MotionEncoder(state_dim, context_dim)
        
        # The Combiner: A GRUCell fuses the two latents
        # Input = new observation, Hidden = prior state memory
        self.combiner = nn.GRUCell(input_size=context_dim, hidden_size=context_dim)
        
        self.ode_func = ODEFunc(state_dim, context_dim, hidden_dim)
        self.criterion = nn.MSELoss()

    def forward(self, x_prev, obs_seq, num_steps=10):
        # 1. Encode the sensor history
        latent_obs = self.obs_encoder(obs_seq)
        
        # 2. Encode the previous state
        latent_state = self.motion_encoder(x_prev)
        
        # 3. FUSE them together using the GRU Cell update
        context = self.combiner(latent_obs, latent_state)
        
        # 4. Integrate the ODE forward in time
        tau_vals = torch.linspace(0.0, 1.0, num_steps + 1, device=self.device)
        dtau = 1.0 / num_steps
        
        x_tau = x_prev
        for i in range(num_steps):
            tau_i = tau_vals[i].item()
            k1 = self.ode_func(tau_i, x_tau, context)
            k2 = self.ode_func(tau_i + dtau/2, x_tau + dtau/2 * k1, context)
            k3 = self.ode_func(tau_i + dtau/2, x_tau + dtau/2 * k2, context)
            k4 = self.ode_func(tau_i + dtau, x_tau + dtau * k3, context)
            
            x_tau = x_tau + (dtau / 6.0) * (k1 + 2*k2 + 2*k3 + k4)
            
        return x_tau

    def training_step(self, batch, batch_idx):
        batch_x, batch_y = batch
        
        # How many steps ahead we want the model to predict on its own
        unroll_steps = 5 
        
        # We start predicting 5 steps before the end of the sequence
        start_t = batch_x.size(1) - unroll_steps 
        
        # 1. Get the initial true state and inject noise
        x_curr = batch_y[:, start_t - 1, :] 
        x_curr = x_curr + (torch.randn_like(x_curr) * self.noise_std)
        
        total_loss = 0.0
        
        # 2. The Unrolled Loop
        for k in range(unroll_steps):
            current_t = start_t + k
            
            # Grab the sensor history up to the current timestep
            # Because GRUs are dynamic, they can handle this growing sequence length!
            obs_seq_k = batch_x[:, :current_t + 1, :]
            
            # The target is the true state at the current timestep
            target_x = batch_y[:, current_t, :]
            
            # PREDICT
            x_next = self(x_prev=x_curr, obs_seq=obs_seq_k)
            
            # Accumulate the loss for this step
            total_loss += self.criterion(x_next, target_x)
            
            # THE CRITICAL STEP: Feed the prediction back in for the next iteration!
            x_curr = x_next
            
        # Average the loss over the unrolled steps
        avg_loss = total_loss / unroll_steps
        
        self.log('train_loss', avg_loss, prog_bar=True)
        return avg_loss

    def validation_step(self, batch, batch_idx):
        batch_x, batch_y = batch
        unroll_steps = 5
        start_t = batch_x.size(1) - unroll_steps 
        
        # No noise injection during validation!
        x_curr = batch_y[:, start_t - 1, :] 
        
        total_loss = 0.0
        
        for k in range(unroll_steps):
            current_t = start_t + k
            obs_seq_k = batch_x[:, :current_t + 1, :]
            target_x = batch_y[:, current_t, :]
            
            x_next = self(x_curr, obs_seq_k)
            total_loss += self.criterion(x_next, target_x)
            x_curr = x_next
            
        avg_loss = total_loss / unroll_steps
        
        self.log('val_loss', avg_loss, prog_bar=True)
        return avg_loss

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.lr, weight_decay=1e-5)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=0.5, patience=3, min_lr=1e-6
        )
        return {
            "optimizer": optimizer,
            "lr_scheduler": {"scheduler": scheduler, "monitor": "val_loss"}
        }

In [ ]:
from torch.utils.data import Dataset, DataLoader
from data_gen import generate_measurements_batched, generate_tracks_batched
import os
import torch
import math
import pytorch_lightning as pl


class MultiTrajectoryDataset(Dataset):
    def __init__(self, measurements, true_states, seq_len=50):
        self.measurements = measurements.transpose(1, 2) 
        self.true_states = true_states.transpose(1, 2)
        self.seq_len = seq_len
        self.windows_per_traj = self.measurements.shape[1] - self.seq_len
        self.total_samples = self.measurements.shape[0] * self.windows_per_traj

    def __len__(self):
        return self.total_samples

    def __getitem__(self, idx):
        traj_idx = idx // self.windows_per_traj
        time_idx = idx % self.windows_per_traj
        x = self.measurements[traj_idx, time_idx : time_idx + self.seq_len, :]
        y = self.true_states[traj_idx, time_idx : time_idx + self.seq_len, :]
        return x, y

class TrackingDataModule(pl.LightningDataModule):
    def __init__(self, num_trajectories=1000, seq_len=50, batch_size=256):
        super().__init__()
        self.num_traj = num_trajectories
        self.seq_len = seq_len
        self.batch_size = batch_size
        
        # Will store normalization stats for un-normalizing later
        self.t_mean = None
        self.t_std = None

    def setup(self, stage=None):
        dim, dt, total_iter = 6, 0.01, 1000
        
        A = torch.zeros((dim, dim), dtype=dtype)
        A[0, 2] = A[1, 3] = A[2, 4] = A[3, 5] = 1.0
        B = torch.zeros((dim, 2), dtype=dtype)
        B[4, 0] = B[5, 1] = 1.0

        R = torch.diag(torch.tensor([0.001**2, 0.001**2], dtype=dtype))
        W = torch.diag(torch.tensor([0.001**2, 0.001**2], dtype=dtype))
        s1 = torch.tensor([[10.0], [0.0]], dtype=dtype)
        s2 = torch.tensor([[30.0], [0.0]], dtype=dtype)
        base_x0 = torch.tensor([[5.0], [20.0], [0.0], [0.0], [2.0], [0.0]], dtype=dtype)

        A1 = torch.eye(dim, dtype=dtype) + (dt * A) + (0.5 * dt**2 * (A @ A))
        noise_scale = math.sqrt(dt) * B @ torch.sqrt(W)

        all_xtrue = torch.zeros((self.num_traj, dim, total_iter), dtype=dtype)
        all_meas = torch.zeros((self.num_traj, 2, total_iter), dtype=dtype)

        # Generate Data
        for traj_idx in range(self.num_traj):
            perturbation = torch.tensor([[2.0], [2.0], [0.5], [0.5], [0.1], [0.1]], dtype=dtype) * torch.randn((dim, 1), dtype=dtype)
            xtrue = torch.zeros((dim, total_iter), dtype=dtype)
            xtrue[:, 0:1] = base_x0 + perturbation
            
            for ii in range(1, total_iter):
                xtrue[:, ii:ii+1] = (A1 @ xtrue[:, ii-1:ii]) + (noise_scale @ torch.randn((2, 1), dtype=dtype))
                
            ps1 = xtrue[0:2, :] - s1
            ps2 = xtrue[0:2, :] - s2
            
            meas_clean = torch.stack([torch.atan2(ps1[1, :], ps1[0, :]), torch.atan2(ps2[1, :], ps2[0, :])], dim=0)
            all_meas[traj_idx] = meas_clean + (torch.sqrt(R) @ torch.randn((2, total_iter), dtype=dtype))
            all_xtrue[traj_idx] = xtrue

        # Split Data (80% Train, 20% Val)
        split_idx = int(self.num_traj * 0.8)
        train_meas, val_meas = all_meas[:split_idx], all_meas[split_idx:]
        train_true, val_true = all_xtrue[:split_idx], all_xtrue[split_idx:]

        # Calculate Statistics (Only on Train)
        m_mean = train_meas.mean(dim=(0, 2), keepdim=True)
        m_std = train_meas.std(dim=(0, 2), keepdim=True) + 1e-8
        self.t_mean = train_true.mean(dim=(0, 2), keepdim=True)
        self.t_std = train_true.std(dim=(0, 2), keepdim=True) + 1e-8

        # Normalize Data
        self.train_meas_norm = (train_meas - m_mean) / m_std
        self.train_true_norm = (train_true - self.t_mean) / self.t_std
        self.val_meas_norm = (val_meas - m_mean) / m_std
        self.val_true_norm = (val_true - self.t_mean) / self.t_std

        # Datasets
        self.train_dataset = MultiTrajectoryDataset(self.train_meas_norm, self.train_true_norm, self.seq_len)
        self.val_dataset = MultiTrajectoryDataset(self.val_meas_norm, self.val_true_norm, self.seq_len)

    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=True, num_workers=4, persistent_workers=True)

    def val_dataloader(self):
        return DataLoader(self.val_dataset, batch_size=self.batch_size, shuffle=False, num_workers=4, persistent_workers=True)

In [ ]:
import matplotlib.pyplot as plt
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from dataset import TrackingDataModule
from model import LitNeuralODEFilter, ModularNeuralODEFilter
import torch
import pytorch_lightning as pl


# 1. Initialize Data and Model
dm = TrackingDataModule(num_trajectories=1000, seq_len=50, batch_size=256)
model = ModularNeuralODEFilter()

# 2. Callbacks for automatic saving and stopping
checkpoint_callback = ModelCheckpoint(monitor='val_loss', mode='min', save_top_k=1, filename='best-ode-filter')
early_stop_callback = EarlyStopping(monitor='val_loss', patience=5, mode='min')

# 3. Train the Model
trainer = pl.Trainer(
    max_epochs=1,
    callbacks=[checkpoint_callback, early_stop_callback],
    gradient_clip_val=1.0, # Keeps ODE solver stable
    accelerator='auto',    # Automatically uses GPU if available
    devices=1
)

print("Starting training...")
trainer.fit(model, datamodule=dm)


# 4. Evaluation and Plotting Phase
print("Evaluating best model on a validation trajectory...")

# FIX 1: Load using the correct class name!
best_model = ModularNeuralODEFilter.load_from_checkpoint(checkpoint_callback.best_model_path)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
best_model = best_model.to(device)
best_model.eval()

traj_idx = 0
seq_len = dm.seq_len
total_steps = dm.val_meas_norm.shape[2]

meas_seq = dm.val_meas_norm[traj_idx].transpose(0, 1)  
true_seq_norm = dm.val_true_norm[traj_idx].transpose(0, 1)

predictions = torch.zeros((total_steps, 6))
predictions[:seq_len] = true_seq_norm[:seq_len] 

# FIX 2: We need a starting physics state to kick off the ODE!
# Grab the true state at the end of the warm-up period.
x_curr = true_seq_norm[seq_len - 1].unsqueeze(0).float().to(device)

with torch.no_grad():
    for t in range(seq_len, total_steps):
        window = meas_seq[t - seq_len + 1 : t + 1].unsqueeze(0).float().to(device)
        
        # FIX 3: Pass both the previous state AND the sensor window
        pred_state = best_model(x_prev=x_curr, obs_seq=window)
        
        # Save the prediction
        predictions[t] = pred_state.squeeze(0).cpu()
        
        # AUTO-REGRESSIVE UPDATE: 
        # The model's current prediction becomes x_curr for the next timestep!
        x_curr = pred_state

# Un-normalize for plotting
t_mean = dm.t_mean.squeeze()
t_std = dm.t_std.squeeze()

predictions_physical = (predictions * t_std) + t_mean
true_physical = (true_seq_norm * t_std) + t_mean

# Plot spatial trajectory
plt.figure(figsize=(10, 8))
plt.plot(true_physical[:, 0].numpy(), true_physical[:, 1].numpy(), label='True Trajectory', color='blue', linewidth=2)
plt.plot(predictions_physical[seq_len:, 0].numpy(), predictions_physical[seq_len:, 1].numpy(), 
         label='Modular Neural ODE Prediction', color='red', linestyle='--', linewidth=2)

plt.scatter([10, 30], [0, 0], marker='^', color='green', s=150, label='Sensors', zorder=5)
plt.title('Modular Latent ODE Filter: Trajectory')
plt.xlabel('X Position')
plt.ylabel('Y Position')
plt.legend()
plt.grid(True)
plt.axis('equal')
plt.savefig('eval_traj.png')